In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType, IntegerType, DateType

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_analysis/resources/utils

In [0]:
products_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_products")
customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_customers")
orders_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_orders")

In [0]:
# ---------------------------------------------------------------------------
# Products → dim_products
# Assumptions:
#   - product_id is the primary key and must not be NULL
#   - Missing categorical fields default to 'unknown'; missing price to 0
# ---------------------------------------------------------------------------

product_schema = StructType([
    StructField('product_id',        StringType(), True),
    StructField('product_name',      StringType(), True),
    StructField('category',          StringType(), True),
    StructField('sub_category',      StringType(), True),
    StructField('state',             StringType(), True),
    StructField('price_per_product', DoubleType(), True),
])

product_df_cleaned = clean_dataframe(
    df=products_df,
    primary_key='product_id',
    fill_defaults={
        'product_name': 'unknown',
        'category': 'unknown',
        'sub_category': 'unknown',
        'state': 'unknown',
        'price_per_product': 0,
    },
    target_schema=product_schema,
)

# Data quality gate
validate_not_null(product_df_cleaned, ['product_id'], label='dim_products')
validate_row_count(product_df_cleaned, min_rows=1, label='dim_products')

delta_writer(product_df_cleaned, 'az_ci_de_dbs.ecom_dev.dim_products')

In [0]:
# ---------------------------------------------------------------------------
# Customers → dim_customers
# Assumptions:
#   - customer_id is the primary key and must not be NULL
#   - All missing text fields default to 'unknown'
# ---------------------------------------------------------------------------

customer_schema = StructType([
    StructField('customer_id',   StringType(), True),
    StructField('customer_name', StringType(), True),
    StructField('email',         StringType(), True),
    StructField('phone',         StringType(), True),
    StructField('address',       StringType(), True),
    StructField('segment',       StringType(), True),
    StructField('country',       StringType(), True),
    StructField('city',          StringType(), True),
    StructField('state',         StringType(), True),
    StructField('postal_code',   StringType(), True),
    StructField('region',        StringType(), True),
])

dim_customers = clean_dataframe(
    df=customers_df,
    primary_key='customer_id',
    fill_defaults={
        'customer_name': 'unknown', 'email': 'unknown', 'phone': 'unknown',
        'address': 'unknown', 'segment': 'unknown', 'country': 'unknown',
        'city': 'unknown', 'state': 'unknown', 'postal_code': 'unknown',
        'region': 'unknown',
    },
    target_schema=customer_schema,
)

# Data quality gate
validate_not_null(dim_customers, ['customer_id'], label='dim_customers')
validate_row_count(dim_customers, min_rows=1, label='dim_customers')

delta_writer(dim_customers, 'az_ci_de_dbs.ecom_dev.dim_customers')

In [0]:
# ---------------------------------------------------------------------------
# Orders → fact_orders
# Assumptions:
#   - order_id is the primary key and must not be NULL
#   - order_date / ship_date use 'd/M/yyyy' format
# ---------------------------------------------------------------------------

fact_orders_schema = StructType([
    StructField('order_id',    StringType(), True),
    StructField('order_date',  DateType(), True),
    StructField('ship_date',   DateType(), True),
    StructField('ship_mode',   StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id',  StringType(), True),
    StructField('quantity',    IntegerType(), True),
    StructField('price',       DoubleType(), True),
    StructField('discount',    DoubleType(), True),
    StructField('profit',      DecimalType(10, 2), True),
])

fact_orders = clean_dataframe(
    df=orders_df,
    primary_key='order_id',
    target_schema=fact_orders_schema,
    date_columns={'order_date': 'd/M/yyyy', 'ship_date': 'd/M/yyyy'},
)

# Data quality gate
validate_not_null(fact_orders, ['order_id', 'customer_id', 'product_id'], label='fact_orders')
validate_row_count(fact_orders, min_rows=1, label='fact_orders')

delta_writer(fact_orders, 'az_ci_de_dbs.ecom_dev.fact_orders')